从零开始实现

定义暂退函数

In [1]:
import torch
from torch import nn

In [6]:
def dropout_layer(X,dropout):
    assert 0<=dropout<=1
    if dropout==1:
        return torch.zeros_like(X)
    if dropout==0:
        return X
    #掩码mask
    mask = (torch.rand(X.shape, device=X.device) > dropout).float()#按照X的形状生成0到1的随机数，如果大于dropout就保留，mask默认会在cpu上，所以要设定它和X的设备相同
    return mask*X/(1.0-dropout)
    

定义模型参数

In [7]:
num_inputs,num_outputs,num_hiddens1,num_hiddens2=784,10,256,256

定义模型

In [8]:
#设置两个隐藏层的drop参数
dropout1,dropout2=0.2,0.5
class Net(nn.Module):
    def __init__(self,num_inputs,num_outputs,num_hiddens1,num_hiddens2,is_training=True):
        super(Net,self).__init__()#初始化父类，让PyTorch能够正常管理网络中的层和参数。
        self.num_inputs=num_inputs
        self.training=is_training
        self.lin1=nn.Linear(num_inputs,num_hiddens1)
        self.lin2=nn.Linear(num_hiddens1,num_hiddens2)
        self.lin3=nn.Linear(num_hiddens2,num_outputs)
        self.relu=nn.ReLU()

    def forward(self,X):
        H1=self.relu(self.lin1(X.reshape((-1,self.num_inputs))))
        #只有在训练模式中启动
        if self.training==True:
            H1=dropout_layer(H1,dropout1)
        H2=self.relu(self.lin2(H1))
        if self.training==True:
            H2=dropout_layer(H2,dropout2)

        out=self.lin3(H2)
        return out

net=Net(num_inputs,num_outputs,num_hiddens1,num_hiddens2)

训练和测试:d2l.train_ch3(net,train_iter,test_iter,loss,num_epochs,trainer)

In [9]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 1. 加载 Fashion-MNIST 数据集
transform = transforms.ToTensor()

train_data = datasets.FashionMNIST(
    root="../data",
    train=True,
    transform=transform,
    download=False
)

test_data = datasets.FashionMNIST(
    root="../data",
    train=False,
    transform=transform,
    download=False
)

batch_size = 256

train_iter = DataLoader(
    train_data, batch_size=batch_size, shuffle=True
)
test_iter = DataLoader(
    test_data, batch_size=batch_size, shuffle=False
)

# 2. 设置训练参数
num_epochs = 10
lr = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = net.to(device)

loss = nn.CrossEntropyLoss()
trainer = torch.optim.SGD(net.parameters(), lr=lr)

# 3. 训练与测试
for epoch in range(num_epochs):
    net.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for X, y in train_iter:
        X, y = X.to(device), y.to(device)

        trainer.zero_grad()
        y_hat = net(X)
        l = loss(y_hat, y)
        l.backward()
        trainer.step()

        total_loss += l.item() * y.shape[0]
        total_correct += (y_hat.argmax(dim=1) == y).sum().item()
        total_samples += y.shape[0]

    train_loss = total_loss / total_samples
    train_acc = total_correct / total_samples

    net.eval()
    test_correct = 0
    test_samples = 0

    with torch.no_grad():
        for X, y in test_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)

            test_correct += (y_hat.argmax(dim=1) == y).sum().item()
            test_samples += y.shape[0]

    test_acc = test_correct / test_samples

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Test Acc: {test_acc:.4f}"
    )

Epoch [1/10] Loss: 0.9114 Train Acc: 0.6631 Test Acc: 0.7887
Epoch [2/10] Loss: 0.5390 Train Acc: 0.8017 Test Acc: 0.7309
Epoch [3/10] Loss: 0.4685 Train Acc: 0.8277 Test Acc: 0.8232
Epoch [4/10] Loss: 0.4280 Train Acc: 0.8438 Test Acc: 0.8374
Epoch [5/10] Loss: 0.4048 Train Acc: 0.8500 Test Acc: 0.8181
Epoch [6/10] Loss: 0.3863 Train Acc: 0.8590 Test Acc: 0.8571
Epoch [7/10] Loss: 0.3692 Train Acc: 0.8644 Test Acc: 0.8373
Epoch [8/10] Loss: 0.3563 Train Acc: 0.8678 Test Acc: 0.8447
Epoch [9/10] Loss: 0.3496 Train Acc: 0.8719 Test Acc: 0.8496
Epoch [10/10] Loss: 0.3378 Train Acc: 0.8756 Test Acc: 0.8727


简洁实现：我们只需要再每个全连接层后面添加一个暂退层

In [11]:
net=nn.Sequential(nn.Flatten(),nn.Linear(784,256),nn.ReLU(),nn.Dropout(dropout1),nn.Linear(256,256),nn.ReLU(),nn.Dropout(dropout2),nn.Linear(256,10))
#初始化权重
def init_weights(m):
    if type(m)==nn.Linear:
        nn.init.normal_(m.weight,std=0.01)
net.apply(init_weights);

训练部分同上

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = net.to(device)

loss = nn.CrossEntropyLoss()
trainer = torch.optim.SGD(net.parameters(), lr=lr)


for epoch in range(num_epochs):
    net.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for X, y in train_iter:
        X, y = X.to(device), y.to(device)

        trainer.zero_grad()
        y_hat = net(X)
        l = loss(y_hat, y)
        l.backward()
        trainer.step()

        total_loss += l.item() * y.shape[0]
        total_correct += (y_hat.argmax(dim=1) == y).sum().item()
        total_samples += y.shape[0]

    train_loss = total_loss / total_samples
    train_acc = total_correct / total_samples

    net.eval()
    test_correct = 0
    test_samples = 0

    with torch.no_grad():
        for X, y in test_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X)

            test_correct += (y_hat.argmax(dim=1) == y).sum().item()
            test_samples += y.shape[0]

    test_acc = test_correct / test_samples

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Test Acc: {test_acc:.4f}"
    )

Epoch [1/10] Loss: 1.1960 Train Acc: 0.5381 Test Acc: 0.7334
Epoch [2/10] Loss: 0.5843 Train Acc: 0.7828 Test Acc: 0.8153
Epoch [3/10] Loss: 0.4906 Train Acc: 0.8203 Test Acc: 0.8362
Epoch [4/10] Loss: 0.4522 Train Acc: 0.8356 Test Acc: 0.8296
Epoch [5/10] Loss: 0.4216 Train Acc: 0.8475 Test Acc: 0.8425
Epoch [6/10] Loss: 0.3937 Train Acc: 0.8579 Test Acc: 0.8346
Epoch [7/10] Loss: 0.3814 Train Acc: 0.8605 Test Acc: 0.8314
Epoch [8/10] Loss: 0.3670 Train Acc: 0.8666 Test Acc: 0.8604
Epoch [9/10] Loss: 0.3591 Train Acc: 0.8680 Test Acc: 0.8590
Epoch [10/10] Loss: 0.3508 Train Acc: 0.8713 Test Acc: 0.8658
